In [114]:
import os
os.chdir('/fast/pmayilvahanan/post_training/dapo/verl/notebooks/')
from utils import hf_tokenizer
import torch
from utils import compute_pass_at_k, load_advantages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [116]:
base_dir = '/fast/pmayilvahanan/post_training/verl_checkpoints/self_distillation_neurips/Qwen/'
checkpoints = {
    'qwen': os.path.join(base_dir, 'qwen_2.5_math_1.5b_base_sft_data_Qwen2.5-Math-1.5B_dsr_sub_n_8_bsz_64_epochs_1_kl_coef_0.0_step_0_epochs_13_lr_1e-5_bsz_128_micro_bsz_128_total_training_steps_70'),
}


In [120]:
split = 'train'
advantages = {}
epochs = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65]
for adv_type in [split]:
    for epoch in epochs:
        if epoch == -1:
            epoch = None
        advantages[epoch] = load_advantages(checkpoints, split=adv_type, epoch=epoch)

Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen
Loaded train advantages for qwen


In [121]:
accuracy = {}
scores = {}
init_indices = []
for epoch in epochs:
    accuracy[epoch] = []
    for idx in advantages[0]['qwen']['samples'].keys():
        if idx not in scores:
            scores[idx] = []
        scores[idx].append(np.mean([rollout['score'] for rollout in advantages[epoch]['qwen']['samples'][idx]]))
        if epoch == 0:
            if scores[idx][0] > 0.0:
                init_indices.append(idx)
        accuracy[epoch].append(np.mean([rollout['score'] for rollout in advantages[epoch]['qwen']['samples'][idx]]))

for idx in scores.keys():
    scores[idx] = np.array(scores[idx])

for epoch in epochs:
    accuracy[epoch] = np.mean(accuracy[epoch])

In [122]:
accuracy

{0: 0.23687066,
 5: 0.26236978,
 10: 0.2999132,
 15: 0.30718315,
 20: 0.33691406,
 25: 0.35134548,
 30: 0.36653647,
 35: 0.37141928,
 40: 0.3876953,
 45: 0.38487414,
 50: 0.3905165,
 55: 0.38693577,
 60: 0.39127603,
 65: 0.38845485}

In [123]:
variance = {}
for idx in init_indices:
    variance[idx] = np.var(scores[idx])

# Sort variance by values
sorted_variance_dict = {k: v for k, v in sorted(variance.items(), key=lambda item: item[1])}
sorted_variance = [(k,v) for k, v in sorted(variance.items(), key=lambda item: item[1])]

# Print key and values
for item in sorted_variance:
    print(f"Index: {item[0]}, Variance: {item[1]}")

max_variance = max(variance.values())
max_variance_idx = max(variance, key=variance.get)

Index: 971, Variance: 0.00103635189589113
Index: 1033, Variance: 0.00103635189589113
Index: 1089, Variance: 0.00103635189589113
Index: 977, Variance: 0.0010363522451370955
Index: 114, Variance: 0.0019132652087137103
Index: 147, Variance: 0.0019132652087137103
Index: 330, Variance: 0.0019132652087137103
Index: 1002, Variance: 0.0019132652087137103
Index: 1073, Variance: 0.0019132652087137103
Index: 947, Variance: 0.0026307397056370974
Index: 196, Variance: 0.002630739938467741
Index: 553, Variance: 0.002630739938467741
Index: 559, Variance: 0.002630739938467741
Index: 874, Variance: 0.002630739938467741
Index: 978, Variance: 0.002630739938467741
Index: 270, Variance: 0.003188775386661291
Index: 508, Variance: 0.0031887756194919348
Index: 727, Variance: 0.0031887756194919348
Index: 82, Variance: 0.0035873723682016134
Index: 439, Variance: 0.0035873723682016134
Index: 1148, Variance: 0.0035873723682016134
Index: 622, Variance: 0.003587372601032257
Index: 797, Variance: 0.00358737260103225

In [124]:
sorted_variance[-10:]

[(551, 0.0785236),
 (897, 0.07948023),
 (749, 0.0867347),
 (1063, 0.0880102),
 (606, 0.088408805),
 (217, 0.092554204),
 (879, 0.0928731),
 (766, 0.09861289),
 (186, 0.10180164),
 (470, 0.104352675)]

In [113]:
idx = 470
advantages[0]['qwen']['samples'][idx]

[{'prompt': "system\nPlease reason step by step, and put your final answer within \\boxed{}.\nuser\nThe year 2009 has a unique property: by rearranging the digits of the number 2009, it is impossible to form a smaller four-digit number (numbers do not start with zero). In which future year will this property first repeat again? Let's think step by step and output the final answer within \\boxed{}.\nassistant\nThe property of the year 2009 is that by rearranging the digits, it is impossible to form a smaller four-digit number. The only way to ensure that a four-digit number cannot be formed smaller is to have the digits arranged in the highest possible order. The digits of the year 2009 are 2, 0, 0, and 9, arranged in descending order as 9802. Rearranging any smaller number would result in a smaller four-digit number. The next year where this property repeats will be the next year after 2009 with the digits 9802 arranged in descending order in the highest possible way. The only year aft

In [56]:
epoch = 0
idx = 124
print(advantages[epoch]['qwen']['samples'][idx][0]['prompt'])





system
Please reason step by step, and put your final answer within \boxed{}.
user
The pressure \( P \) exerted by wind on a sail varies jointly as the area \( A \) of the sail and the cube of the wind's velocity \( V \). When the velocity is \( 8 \) miles per hour, the pressure on a sail of \( 2 \) square feet is \( 4 \) pounds. Find the wind velocity when the pressure on \( 4 \) square feet of sail is \( 32 \) pounds. Let's think step by step and output the final answer within \boxed{}.
assistant
The pressure \( P \) exerted by wind on a sail is given by the equation:
\[ P = k \cdot A \cdot V^3 \]
where \( k \) is a constant. We first need to find the constant \( k \):
\[ 4 = k \cdot 2 \cdot 8^3 \]
After solving for \( k \), we can then find the velocity \( V \) when the pressure \( P \) is 32 pounds on a 4 square feet sail using the equation:
\[ 32 = k \cdot 4 \cdot V^3 \]

Let's start by solving for \( k \):
\[ k = \frac{4}{2 \cdot 8^3} = \frac{4}{2 \cdot 512} = \frac{4}{1024} = \f